<a href="https://colab.research.google.com/github/zFonta/CEIA-TF-Chess-DL/blob/main/notebooks/00_survey_dumps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 - Relevamiento de los dumps de Lichess

**Objetivo:** medir cuantas partidas del dump mensual sobreviven al filtro
(ELO >= 2200 a ambos jugadores, controles Blitz / Rapid / Classical) **antes** de
gastar horas de computo en el etiquetado con Stockfish.

Este paso es barato: recorre el dump por streaming y solo cuenta. No descarga el
archivo entero ni invoca al motor. Con el numero que devuelve se decide si
alcanza con un solo mes o si hay que sumar dumps.

> **Runtime:** usar **CPU**, no GPU. Este notebook no entrena nada.

Corresponde a la tarea 3.1 del WBS (descarga y exploracion de partidas).

## 1. Entorno

In [3]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija (queda registrada en cada fila del dataset).
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

Listo. Directorio de trabajo: /content/CEIA-TF-Chess-DL


In [4]:
from chessdl.colab import describe_runtime

runtime = describe_runtime("stockfish")
print(runtime.summary())

Colab runtime : True
CPU workers   : 2
GPU present   : False
Stockfish     : /content/CEIA-TF-Chess-DL/bin/stockfish


## 2. Configuracion

Todos los parametros del pipeline salen de `configs/dataset_v1.yaml`, que es la
unica fuente de verdad y se versiona junto con el codigo (requerimiento 2.3).

In [5]:
from chessdl.config import load_config

cfg = load_config()
print("Dumps configurados :", cfg.source.dumps)
print("ELO minimo         :", cfg.filter.min_elo, "(exigido a ambos jugadores)")
print("Controles de tiempo:", cfg.filter.time_controls)
print("Posiciones/partida :", cfg.sampling.positions_per_game)

Dumps configurados : ('2025-06',)
ELO minimo         : 2200 (exigido a ambos jugadores)
Controles de tiempo: ('Blitz', 'Rapid', 'Classical')
Posiciones/partida : 4


## 3. Relevamiento

`--max-scanned` limita cuantas partidas se **miran**. Un millon de partidas
alcanza para estimar la tasa de aceptacion con buena precision y tarda pocos
minutos; sin el limite recorre el dump completo.

Nota: la tasa estimada sobre el principio del archivo puede sesgarse levemente,
porque el dump viene ordenado por fecha dentro del mes.

In [6]:
from chessdl.data import pipeline

resultados = pipeline.survey(cfg, max_scanned=1_000_000)

for stats in resultados:
    print(stats.summary())

accepted from lichess_db_standard_rated_2025-06.pgn.zst: 0game [00:00, ?game/s]

2025-06: 1,000,072 games seen, 17,595 accepted (1.759%), 0 written


### La misma operación desde la línea de comando

Las celdas de arriba usan la API de Python. El repositorio expone además un
comando equivalente, que es el que documenta el README como vía de reproducción
(requerimiento 2.2). Desde una celda se invoca con `!`:

In [7]:
!{sys.executable} -m chessdl.scripts.build_dataset --survey --max-scanned 200000 --quiet


Survey of the configured dumps
------------------------------------------------------------
2025-06: 200,130 games seen, 3,704 accepted (1.851%), 0 written
------------------------------------------------------------
3,704 usable games -> about 14,816 positions


## 4. Cuantos meses hacen falta

Extrapolamos de la tasa medida al dump completo. Un dump mensual de Lichess
ronda las 90-100 millones de partidas.

In [8]:
PARTIDAS_POR_DUMP = 95_000_000   # orden de magnitud de un mes de Lichess
OBJETIVO_POSICIONES = 2_000_000  # objetivo para entrenar la ResNet

tasa = sum(s.games_accepted for s in resultados) / max(sum(s.games_seen for s in resultados), 1)
partidas_por_mes = PARTIDAS_POR_DUMP * tasa
posiciones_por_mes = partidas_por_mes * cfg.sampling.positions_per_game

print(f"Tasa de aceptacion medida : {tasa:.4%}")
print(f"Partidas utiles por mes   : {partidas_por_mes:,.0f}")
print(f"Posiciones por mes        : {posiciones_por_mes:,.0f}")
print()
meses = max(1, round(OBJETIVO_POSICIONES / max(posiciones_por_mes, 1) + 0.49))
print(f"Para {OBJETIVO_POSICIONES:,} posiciones hacen falta ~{meses} mes(es) de dump.")

Tasa de aceptacion medida : 1.7594%
Partidas utiles por mes   : 1,671,405
Posiciones por mes        : 6,685,619

Para 2,000,000 posiciones hacen falta ~1 mes(es) de dump.


Si hace falta mas de un mes, agregar los meses adicionales a `source.dumps`
en `configs/dataset_v1.yaml`. El pipeline los procesa en orden y guarda el
progreso por `(dump, offset)`, asi que se pueden sumar meses despues sin
rehacer nada de lo ya etiquetado.

**Proximo paso:** `01_build_dataset.ipynb`.